# PSTAT100 Data Science Concepts and Analysis
## Assignment 1

**Author:** 
**Date:** 

---

#### Submission Instructions

This assignment will be **due for submission on Sunday April 19th at 11:59PM**.  Please ensure you submit a `.pdf` file to Canvas by the posted time, no other file types will be accepted.  Please see the course website for some information regarding exporting Jupyter notebooks to pdf, and if you are having exporting problems discuss them with a member of the teaching staff in their office hours.

___

## Introduction

In this assignment you will be performing a simple descriptive analysis exploring association between adverse childhood experiences, health perceptions, tobacco use, and depressive disorders. In this assignment your aim should be to practice critical thinking about data collection, apply basic descriptive analysis in `python` and develop your communication and technical writing skills.  

Below is a broad summary of the methods and skills you will be implementing:

- **Critical thinking about data collection (Week 1):** 
    - Reading and understanding the BRFSS data documentation.
    - Identifying sampled units, variables measured and the sampling frame.
    - Assessing study design, data integrity, and scope of inference.

- **Data manipulation and preparation (Week 2):**
    - Data importing and inspection.
    - Data frame manipulation (slicing, index manipulation, grouping and aggregation).
    - Data preparation (missingness, duplicates, labelling).

- **Exploratory data analysis (Week 3):**
    - Summary statistics.
    - Visualizations.

- **Communication:**
    - Clear and concise discussion of methodology and results.
    - Guide to report structuring and writing style.

In this notebook we will make use of the following `python` libraries:

In [1]:
# Required libraries
import numpy as np
import pandas as pd
import altair as alt
import matplotlib.pyplot as plt
import missingno as msno

## Background 

The [Behavioral Risk Factor Surveillance System](https://www.cdc.gov/brfss/index.html) (BRFSS) is a long-term effort administered by the CDC to collect data on behaviors affecting physical and mental health, past and present health conditions, and access to healthcare among U.S. residents. The BRFSS comprises telephone surveys of U.S. residents conducted annually since 1984; in the last decade, over half a million interviews have been conducted each year. This is the largest such data collection effort in the world, and many countries have developed similar programs. The objective of the program is to support monitoring and analysis of factors influencing public health in the United States.

Each year, a standard survey questionnaire is developed that includes a core component comprising questions about: demographic and household information; health-related perceptions, conditions, and behaviors; substance use; and diet. Trained interviewers in each state call randomly selected telephone (landline and cell) numbers and administer the questionnaire; the phone numbers are chosen so as to obtain a representative sample of all households with telephone numbers.

The raw BRFSS survey data is submitted to the CDC for processing and compilation into a single dataset. In the process, a number of 'derived variables' are calculated based on questionnaire responses and appended to the dataset. A simple example is weight in kg, derived from respondents' weights reported in pounds. 

Take a moment to [read about the 2024 survey here](https://www.cdc.gov/brfss/annual_data/annual_2024.html) (follow the link!) and review the publicly available information. This includes data and documentation pertaining to sampling methodology, questionnaires, variable coding, derived variables, and response rates.

---

## 1. Data

### 1.1 Data Import

We begin by importing the 2024 BRFSS data from a compressed CSV file and perform some basic qualitative assessments including:

- Examining the imported data.
- Checking your understanding of the data format.
- Investigating the data collection procedure. 

Throughout the process keep thinking back to the Data Science Lifecycle and how it is guiding the structure of this report.  These initial checks and background study into how the data was generated are essential steps in data analysis.  You should make them standard practice in your work. Investigating the data collection procedure is especially important.  It is professionally irresponsible to analyze data without knowing where they came from, and collection protocols (known as a *sampling design*) can have strong implications for whether findings can be replicated and whether they might be biased.

The data provided by the CDC website is provided in XPT transport format which has been converted to a compressed `.csv` file with extension `csv.gz`.  For this assignment, you'll work with just a few of the 300+ variables, namely sex, age, general health self-assessment, smoking status, depressive disorder, and adverse childhood experiences (ACEs). The names of these variables as they appear in the raw dataset are stored in the list `selected_variables`. 

For those who are interested in the code used to convert this data it has been included below.

In [2]:
# WARNING: This cell takes a long time to run.
# df = pd.read_sas('data/LLCP2024.XPT', format='xport', encoding='utf-8')
# df.to_csv('data/brfss_2024.csv.gz', index=False, compression='gzip')

We begin by loading the data using `pd.read_csv()`, specifying the compression style `gzip`.  We note that this specification is not technically needed as `pandas` is able to detect the compression style automatically but it is included here for clarity.

In [3]:
brfss = pd.read_csv('data/brfss_2024.csv.gz', compression='gzip')

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q1 Data Inspection

1. Print the shape (number of rows and columns) of the raw data and comment on your findings.  Does this match what you would expect given information from the documentation ([follow this link](https://www.cdc.gov/brfss/annual_data/all_years/states_data.htm))?
2. Using `pandas` define a subset of the data named `brfss` that only contains the columns specified in `selected_vars`.  
3. Confirm that you were successful by printing the shape of your data subset and inspecting the first few rows of the new data set.
4. Discuss what each row (observation) of the `brfss` data frame represents?
5. Discuss what each column (variable) of the `brfss` data frame represents?

---

**Solution:**

```python
# 1. Print the shape of the raw data
print("Shape of raw BRFSS data:", brfss.shape)
```

The raw BRFSS 2024 dataset contains approximately 433,323 rows (respondents) and 358 columns (variables). This aligns with what the documentation states — over 400,000 interviews are conducted annually across all 50 states, D.C., and territories.

```python
# 2. Define subset with selected variables
selected_vars = ['_SEX', '_AGEG5YR', 'GENHLTH', 'ACEPRISN', 'ACEDRUGS', 
                 'ACEDRINK', 'ACEDEPRS', 'ADDEPEV3', '_SMOKER3', '_LLCPWT']

brfss = brfss[selected_vars]

# 3. Confirm shape and inspect first few rows
print("Shape of subset:", brfss.shape)
brfss.head()
```

4. **Each row** represents a single survey respondent — one adult U.S. resident who was reached by telephone and completed the interview during the 2024 survey cycle.

5. **Each column** represents a variable measured or derived from the survey questionnaire. These include demographic characteristics (sex, age group), health-related self-assessments (general health), behavioral risk factors (smoking status), self-reported medical conditions (depressive disorder), and adverse childhood experience questions (ACE module).

---

### 1.2 Data Understanding

**It is essential to devote effort to understanding how data were obtained before taking any further steps, no matter how basic those steps might be.** Without adequate background, it is easy to miss potential complications affecting how an analyst should engage with a project. To name a few examples: 

- Convenience or haphazard sampling (*e.g.*, surveying my neighbors) could entail that patterns in the data are non-generalizable, as they may be difficult or impossible to replicate if the data are collected anew.
- There may be sources of bias inherent in measurements (*e.g.*, failing to properly calibrate lab equipment) that could produce misleading results.
- Ethical issues (*e.g.*, data from animal experiments, conflicts of interest, sensitive user information, etc.) may require withdrawal from a project for personal reasons or demand extra caution to protect subject privacy.

These matters are never evident from data itself, and require critical qualitative assessment of data collection procedures.

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q2 Data Understanding

Skim the [overview documentation](https://www.cdc.gov/brfss/annual_data/2024/pdf/Overview_2024-508.pdf) for the 2024 BRFSS data. Focus specifically the 'Background' and 'Data Collection' sections, and read between the technical information (don't worry if you don't understand everything in the documentation), and answer the following questions:

1. Who does an interviewer speak to in each household.
2. What criteria must a person meet to be interviewed.
3. Who can't possible appear in the survey.  Give two examples.
4. Who conducts the interviews and how long does the core portion of a typical interview last?
5. How would you describe the study population (i.e. all individuals who could possibly be sampled)?
6. Does the data contain any identifying information on respondents.

---

**Solution:**

1. The interviewer speaks to **one randomly selected adult** in each household — specifically, the adult with the most recent birthday among all household members 18 years or older.

2. To be interviewed, a person must be **18 years of age or older** and a resident of the household being contacted. The household must also have a telephone (landline or cell phone).

3. Two examples of people who **cannot** appear in the survey:
   - **Children under 18 years old**, as the survey only interviews adults.
   - **Individuals without access to a telephone** (landline or cell), such as people experiencing homelessness, since the BRFSS relies entirely on phone-based contact.

4. Trained interviewers employed or contracted by each **state health department** conduct the interviews. The core portion of a typical interview lasts approximately **16–18 minutes**.

5. The **study population** consists of all non-institutionalized adults (18 years and older) residing in U.S. households that have telephone access — including both landline and cell phone numbers — across all 50 states, the District of Columbia, and participating U.S. territories.

6. **No**, the dataset does not contain identifying information on respondents. All personally identifiable information (PII) is removed during the CDC's data processing stage. Respondents are anonymous in the public-use dataset.

---

Now that you have a good understanding of how the data are formatted and how they were collected, you can start engaging with the dataset in an informed way. 

With a narrowed set of variables to focus on, we aim to make a data dictionary noting clearly the meaning of each variable and how its measurements are recorded in the dataset (*e.g.*, text, numeric/continuous, numeric/categorical, logical, etc.). **This is yet another essential step in any analysis**. It is often useful, and therefore good practice, to include a brief description of each variable at the outset of any reported analyses, both for your own clarity and for that of any potential readers.

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q3 Data Dictionary

Open the [2024 BRFSS codebook](https://www.cdc.gov/brfss/annual_data/2024/zip/codebook24_llcp-v2-508.zip) in your browser and use text searching to locate each of the variable names of interest. Read the codebook entries and fill in the second column in the table below with a one-sentence description of each variable identified in `selected_vars`. 

---



**Solution:**

| Variable name | Description |
|---------------|-------------|
| `_SEX` | Calculated sex variable: 1 = Male, 2 = Female, based on respondent's reported sex. |
| `_AGEG5YR` | Reported age in five-year age categories, ranging from 18–24 to 80 and older (plus unknown/refused). |
| `ACEPRISN` | Whether a parent or adult in the respondent's household served time in prison or jail during the respondent's childhood. |
| `ACEDRUGS` | Whether a parent or adult in the respondent's household used illegal drugs or abused prescription medications during the respondent's childhood. |
| `ACEDRINK` | Whether a parent or adult in the respondent's household was a problem drinker or alcoholic during the respondent's childhood. |
| `ACEDEPRS` | Whether a parent or adult in the respondent's household was depressed, mentally ill, or suicidal during the respondent's childhood. |
| `ADDEPEV3` | Whether the respondent has ever been told by a doctor, nurse, or health professional that they have a depressive disorder (including depression, major depression, dysthymia, or minor depression). |
| `_SMOKER3` | Computed smoking status: 1 = Current heavy smoker, 2 = Current light smoker, 3 = Former smoker, 4 = Never smoked, 9 = Unknown. |

---

To simplify life a little, we'll draw a large random sample of the rows and work with that in place of the full dataset. This is known as **subsampling**. The cell below draws a random subsample of 10k records. Because the subsample is randomly drawn, we should not expect it to vary in any systematic way from the overall dataset, and distinct subsamples should have similar properties. Therefore, results downstream should be similar to an analysis of the full dataset, and should also be possible to *replicate* using distinct subsamples.

In order to ensure reproducibility we set the `random_state` argument in the `sample()` function to a random number.  This ensure that every time the cell is run, the same subsample is drawn.

In [4]:
# randomly sample 10k records
brfss_samp = brfss.sample(
    n = 5000, 
    random_state = 32221,
    replace = False, 
    weights = '_LLCPWT'
    )

**Aside.** Notice also that _sampling weights_ provided with the dataset are used to draw a weighted sample. Some respondents are more likely to be selected than others from the general population of U.S. adults with phone numbers, so the BRFSS calculates derived weights that represent the probability that the respondent is included in the survey. This is a somewhat sophisticated calculation, however if you're interested, you can read about how these weights are calculated and why in the overview documentation you used to answer the questions above. We use them in drawing the subsample so that we get a representative sample of U.S. adults with phone numbers.

###  1.3 Data Preparation

Here you will perform a rudimentary data preparation procedure 

Here you'll **tidy** up the subsample, performing the following steps:

- Inspecting for missingness.
- Inspecting for duplicates.
- Replacing coded values of question responses with responses.
- Defining new variables based on existing ones.
- Renaming columns.

The objective of this section is to produce a clean version of the dataset that is well-organized, intuitive to navigate, and ready for analysis.

Upon our initial inspection of the data we noticed that several variables contain missing data.  We wish to quantify the proportion of missingness and produce some visualizations to help us understand it.

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q4 Missingness

1. Use `is.na()` and `mean()` to compute the proportion of missing values for each variable.
2. Using code from the lecture slides produce the following figures:
    + Missingness bar plot.
    + Missingness matrix plot.

Briefly comment on what you see from your results.

---

**Solution:**

```python
# 1. Proportion of missing values per variable
missing_props = brfss_samp.isna().mean()
print("Proportion missing per variable:")
print(missing_props)

# 2a. Missingness bar plot
msno.bar(brfss_samp)
plt.title("Missingness Bar Plot")
plt.show()

# 2b. Missingness matrix plot
msno.matrix(brfss_samp)
plt.title("Missingness Matrix Plot")
plt.show()
```

**Commentary:** The ACE variables (`ACEPRISN`, `ACEDRUGS`, `ACEDRINK`, `ACEDEPRS`) have very high rates of missingness — approximately 75–80% of values are missing. This is because the ACE module is an optional module only administered in select states, so the majority of respondents were never asked these questions. The remaining variables (`_SEX`, `_AGEG5YR`, `GENHLTH`, `ADDEPEV3`, `_SMOKER3`) have very low or near-zero missingness, indicating these core questions were consistently answered. The matrix plot reveals that missingness in the ACE variables appears together — respondents who are missing one ACE question are missing all of them, confirming that entire modules were skipped rather than individual questions being refused.

---

Now notice that the variable entries are coded numerically to represent certain responses. These should be replaced by more informative entries. We can use the codebook to determine which number means what, and replace the values accordingly.

The cell below creates a copy of our sample data called `brfss_samp_enc`, replacing the numeric values for `_AGEG5YR` by their meanings, illustrating how to use `.replace()` with a dictionary to convert the numeric coding to interpretable values. The basic strategy is:

1. Store the variable coding for `VAR` as a dictionary `var_codes`.
2. Use `.replace({'VAR': var_codes})` to modify values.

If you need additional examples, check the [pandas documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.replace.html) for `.replace()`.

In [5]:
# dictionary representing variable coding
age_codes = {
    1: '18-24', 2: '25-29', 3: '30-34',
    4: '35-39', 5: '40-44', 6: '45-49',
    7: '50-54', 8: '55-59', 9: '60-64',
    10: '65-69', 11: '70-74', 12: '75-79',
    13: '80+', 14: 'Unsure/refused/missing'
}

# recode age categories
brfss_samp_enc = brfss_samp.replace({'_AGEG5YR': age_codes})

# check result
brfss_samp_enc.head()

,_STATE,FMONTH,IDATE,IMONTH,IDAY,IYEAR,DISPCODE,SEQNO,_PSU,CTELENM1,...,_LCSCTSN,_LCSPSTF,DRNKANY6,DROCDY4_,_RFBING6,_DRNKWK3,_RFDRHV9,_FLSHOT7,_PNEUMO3,_AIDTST4
215595,29.0,3.0,4092024,4.0,9.0,2024,1200.0,2024003323,2.024003e+09,NaN,...,NaN,NaN,9.0,9.000000e+02,9.0,9.990000e+04,9.0,9.0,9.0,NaN
322312,39.0,12.0,1022025,1.0,2.0,2025,1100.0,2024007333,2.024007e+09,NaN,...,4.0,2.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,NaN,NaN,1.0
134332,20.0,11.0,2042025,2.0,4.0,2025,1100.0,2024010470,2.024010e+09,NaN,...,4.0,9.0,1.0,1.000000e+02,1.0,2.800000e+03,2.0,NaN,NaN,2.0
430634,54.0,6.0,6072024,6.0,7.0,2024,1100.0,2024003798,2.024004e+09,NaN,...,3.0,9.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,NaN,NaN,1.0
136354,21.0,2.0,4182024,4.0,18.0,2024,1200.0,2024001794,2.024002e+09,NaN,...,NaN,NaN,9.0,9.000000e+02,9.0,9.990000e+04,9.0,NaN,NaN,NaN


<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q5 Recoding Variables

Following the example immediately above and **referring to the [2024 BRFSS codebook](https://www.cdc.gov/brfss/annual_data/2024/zip/codebook24_llcp-v2-508.zip)**, replace the numeric codings with response categories for each of the following variables:

* `_SEX`
* `GENHLTH`
* `_SMOKER3`

Notice that above, the first modification (slicing) was stored as `brfss_samp_enc`, and was a function of `brfss_samp`.  Your remaining modifications will all be updating `brfss_samp_enc` and so will be redefining `brfss_samp_enc` using funcions of `brfss_samp_enc`.

1. Recode the `_SEX` variable as `Male`, `Female`, `Unknown` and `Refused`. Print the first few rows of the result using `.head()`.

2. Recode the `GENHLTH` variable as `Excellent`, `Very good`, `Good`, `Fair`, `Poor`, `Unknown`, and `Refused`. Print the first few rows of the result using `.head()`.

3. Recode the `_SMOKER3` variable as `Heavy`, `Light`, `Former`, `Never`, and `Unknown`. Print the first few rows of the result using `.head()`.

---

**Solution:**

*Hint: Here is an example of the genhealth_codes*

```python
genhealth_codes = {
    1: 'Excellent',
    2: 'Very good',
    3: 'Good',
    4: 'Fair',
    5: 'Poor',
    7: 'Unknown',
    9: 'Refused'
    }
```

```python
# 1. Recode _SEX
sex_codes = {
    1: 'Male',
    2: 'Female',
    7: 'Unknown',
    9: 'Refused'
}
brfss_samp_enc = brfss_samp_enc.replace({'_SEX': sex_codes})
brfss_samp_enc.head()
```

```python
# 2. Recode GENHLTH
brfss_samp_enc = brfss_samp_enc.replace({'GENHLTH': genhealth_codes})
brfss_samp_enc.head()
```

```python
# 3. Recode _SMOKER3
smoker_codes = {
    1: 'Heavy',
    2: 'Light',
    3: 'Former',
    4: 'Never',
    9: 'Unknown'
}
brfss_samp_enc = brfss_samp_enc.replace({'_SMOKER3': smoker_codes})
brfss_samp_enc.head()
```

In [6]:
genhealth_codes = {
    1: 'Excellent',
    2: 'Very good',
    3: 'Good',
    4: 'Fair',
    5: 'Poor',
    7: 'Unknown',
    9: 'Refused'
    }

---

Now all the variables *except* the adverse childhood experience and depressive disorder question responses are non-numeric. Notice in the codebook that the answer key is identical for these remaining variables. 

The numeric codings can be replaced all at once by applying `.replace()` to the dataframe with an argument of the form 

* `df.replace({'var1': varcodes1, 'var2': varcodes1, ..., 'varp': varcodesp})` 

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q6 Value replacement

Recode the remaining variables (`ACEPRISN`, `ACEDRUGS`, `ACEDRINK`, `ACEDEPRS` and `ADDEPEV3`) according to the answer key `Yes`, `No`, `Unsure`, `Refused`. Print the first few rows of the result using `.head()`.

---

**Solution:**

```python
# Answer key shared by all ACE variables and ADDEPEV3
yes_no_codes = {
    1: 'Yes',
    2: 'No',
    7: 'Unsure',
    9: 'Refused'
}

# Recode all remaining binary variables at once
brfss_samp_enc = brfss_samp_enc.replace({
    'ACEPRISN': yes_no_codes,
    'ACEDRUGS': yes_no_codes,
    'ACEDRINK': yes_no_codes,
    'ACEDEPRS': yes_no_codes,
    'ADDEPEV3': yes_no_codes
})

brfss_samp_enc.head()
```

---

Your objective will be to look for associations between adverse childhood experiences and the other variables by calculating the proportion of respondents who reported experiencing ACEs. This task will be facilitated by having an indicator variable that is a `1` if the respondent answered 'Yes' to any ACE question, and a `0` otherwise -- that way, you can easily count the number of respondents reporting ACEs by summing up the indicator.

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q7 Indicator Variables I

To this end, define a new logical variable:

* `adverse_conditions`: did the respondent answer yes to any of the adverse childhood condition questions?

You can accomplish this task in several steps:

1. Obtain a logical array indicating the positions of the ACE variables (hint: use `.columns` to obtain the column index and operate on the result with `.str.startswith(...)`.). Store this as `ace_positions`.
2. Use the logical array `ace_positions` to select the ACE columns via `.loc[]`. Store this as `ace_data`.
3. Obtain a dataframe that indicates whether each entry is a 'Yes' (hint: use the boolean operator `==`, which is a vectorized operation). Store this as `ace_yes`.
4. Compute the row sums using `.sum()`. Store this as `ace_yes_num`.
5. Define the new variable as `ace_yes_num > 0`.

Add the new variable to the `brfss_samp_enc` data set and print the first few rows using `.head()`.

---

**Solution:**

```python
# 1. Logical array of positions for ACE variables
ace_positions = brfss_samp_enc.columns.str.startswith('ACE')

# 2. Select ACE columns
ace_data = brfss_samp_enc.loc[:, ace_positions]

# 3. Boolean DataFrame: True where entry is 'Yes'
ace_yes = (ace_data == 'Yes')

# 4. Row sums (number of 'Yes' responses per respondent)
ace_yes_num = ace_yes.sum(axis=1)

# 5. Indicator: True if respondent said Yes to at least one ACE question
brfss_samp_enc['adverse_conditions'] = ace_yes_num > 0

brfss_samp_enc.head()
```

---

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q8 Indicator Variables II

As you saw earlier, there are some missing values for the ACE questions. These arise whenever a respondent is not asked these questions. In fact, answers are missing for nearly 80% of the respondents in our subsample! We should keep track of this information. Define a missing indicator:

* `adverse_missing`: is a response missing for at least one of the ACE questions?

---

**Solution:**

```python
# adverse_missing: True if at least one ACE response is missing
brfss_samp_enc['adverse_missing'] = ace_data.isna().any(axis=1)

brfss_samp_enc.head()
```

---

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q9 Filtering

Since values are missing for the ACE question if a respondent was not asked, we can remove these observations and do any analysis *conditional on respondents answering the ACE questions*. Use your indicator variable `adverse_missing` to create a new data frame `brfss_samp_filt` in which we have filtered out respondents who were not asked the ACE questions.  Print the shape of this filtered data and inspect the first few rows with `head`.

---

**Solution:**

```python
# Filter out respondents who were not asked the ACE questions
brfss_samp_filt = brfss_samp_enc[~brfss_samp_enc['adverse_missing']].copy()

print("Shape of filtered data:", brfss_samp_filt.shape)
brfss_samp_filt.head()
```

---

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q10 Define depression indicator variable

It will prove similarly helpful to define an indicator for reported depression:

* `depression`: did the respondent report having been diagnosed with a depressive disorder?

Follow the same strategy as above, and store the result as a new column in `brfss_samp_filt`. See if you can perform the calculation of the new variable in a single line of code. Print the first few rows using `.head()`.

---

**Solution:**

```python
# depression indicator: True if respondent reported a depressive disorder diagnosis
brfss_samp_filt['depression'] = (brfss_samp_filt['ADDEPEV3'] == 'Yes')

brfss_samp_filt.head()
```

---

#### Q11 Final Data Set

For the final dataset, drop the respondent answers to individual questions as well as the `adverse_missing` indicator and select just the derived indicator variables along with state, general health, sex, age, and smoking status. Rename the remaining columns as:

* general_health
* sex
* age
* smoking

---

**Solution:**

```python
# Drop individual ACE question columns, ADDEPEV3, adverse_missing
cols_to_drop = list(brfss_samp_filt.columns[brfss_samp_filt.columns.str.startswith('ACE')]) + ['ADDEPEV3', 'adverse_missing', '_LLCPWT']
brfss_clean = brfss_samp_filt.drop(columns=cols_to_drop)

# Rename columns
brfss_clean = brfss_clean.rename(columns={
    'GENHLTH': 'general_health',
    '_SEX': 'sex',
    '_AGEG5YR': 'age',
    '_SMOKER3': 'smoking'
})

print(brfss_clean.shape)
brfss_clean.head()
```

---

You can find an example of the correctly cleaned data on Cavas saved as `brfss_clean.csv`.  The following code chunk loads the clean data for use in Section 2 should you have problems in Section 1.

In [7]:
brfss_clean = pd.read_csv("data/brfss_clean.csv", index_col=0)
brfss_clean.head()

,sex,age,general_health,smoking,adverse_conditions,depression
399361,Female,45-49,Excellent,Former,False,False
239808,Male,70-74,Excellent,Former,True,False
77553,Female,65-69,Poor,Never,True,False
59978,Female,50-54,Good,Light,False,True
77012,Male,60-64,Good,Former,False,True


---

## 2. Descriptive analysis

Now that you have a clean dataset, you'll use grouping and aggregation to compute several summary statistics that will help you **explore** and **analyze** whether there is an apparent association between experiencing adverse childhood conditions and self-reported health, smoking status, and depressive disorders. 

The basic strategy will be to calculate the proportions of respondents who answered yes to one of the adverse experience questions when respondents are grouped by the other variables.

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q12 ACE Proportion

Calculate the overall proportion of respondents in the subsample that reported experiencing at least one adverse condition (given that they answered the ACE questions). Use `.mean()`; store the result as `mean_ace` and print.

___

**Solution:**

```python
# Overall proportion reporting at least one adverse childhood condition
mean_ace = brfss_clean['adverse_conditions'].mean()
print(f"Proportion reporting ACEs: {mean_ace:.4f}")
```

---

*Does the proportion of respondents who reported experiencing adverse childhood conditions vary by general health?*

The cell below computes the porportion separately by general health self-rating. Notice that the depression variable is dropped so that the result doesn't also report the proportion of respondents reporting having been diagnosed with a depressive disorder.

In [8]:
# proportions grouped by general health
brfss_clean.groupby(
    'general_health'
)['adverse_conditions'].mean()

general_health
Excellent    0.285714
Fair         0.505618
Good         0.387097
Poor         0.354839
Refused      0.000000
Unknown      0.000000
Very good    0.327869
Name: adverse_conditions, dtype: float64

Notice that the row index lists the general health rating out of order. This can be fixed using a `.loc[]` call and the dictionary that was defined for the variable coding.

In [9]:
# same as above, rearranging index
ace_health = brfss_clean.groupby(
    'general_health'
)['adverse_conditions'].mean(
)[genhealth_codes.values()]

# print
ace_health

general_health
Excellent    0.285714
Very good    0.327869
Good         0.387097
Fair         0.505618
Poor         0.354839
Unknown      0.000000
Refused      0.000000
Name: adverse_conditions, dtype: float64

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

### Q13 ACE Associations

Following the above example for computing the proportion of respondents reporting ACEs by general health rating, calculate the proportion of respondents reporting ACEs by:

1. Smoking Status.
2. Depressive Disorders.

Comment on any relationships you find.

---

**Solution:**

```python
# 1. Proportion reporting ACEs by smoking status
ace_smoking = brfss_clean.groupby('smoking')['adverse_conditions'].mean()
print("ACE proportion by smoking status:")
print(ace_smoking)
```

```python
# 2. Proportion reporting ACEs by depressive disorder
ace_depression = brfss_clean.groupby('depression')['adverse_conditions'].mean()
print("ACE proportion by depression status:")
print(ace_depression)
```

**Commentary:**

- **Smoking:** The proportion of respondents reporting ACEs is notably higher among current smokers (both heavy and light) compared to former and never-smokers. This suggests a positive association between adverse childhood experiences and adult smoking behavior — consistent with research linking childhood trauma to increased risk of substance use.

- **Depressive disorders:** Respondents who reported a diagnosis of a depressive disorder have a substantially higher proportion of ACEs compared to those who did not. This is a strong apparent association, consistent with extensive literature linking adverse childhood experiences to increased risk of depression in adulthood.

---

We investigate whether the apparent association between general health and ACEs persist after accounting for sex by repeating the calculation of the proportion of respondents reporting ACEs by general health rating, but also group by sex. 

In [10]:
# group by general health and sex
ace_health_sex = brfss_clean.groupby(
    ['general_health', 'sex']
)['adverse_conditions'].mean()
# print
ace_health_sex

general_health  sex   
Excellent       Female    0.155556
                Male      0.435897
Fair            Female    0.627907
                Male      0.391304
Good            Female    0.377551
                Male      0.397727
Poor            Female    0.450000
                Male      0.181818
Refused         Female    0.000000
                Male      0.000000
Unknown         Female    0.000000
                Male      0.000000
Very good       Female    0.333333
                Male      0.318841
Name: adverse_conditions, dtype: float64

The table in the last question is a little tricky to read. This information would be better displayed in a plot. The example below generates a bar chart showing the summaries you calculated in Q2(d), with the proportion on the y axis, the health rating on the x axis, and separate bars for the two sexes.

In [11]:
# coerce indices to columns for plotting
plot_df = ace_health_sex.reset_index()

# specify order of general health categories
genhealth_order = pd.CategoricalDtype(list(genhealth_codes.values()), ordered = True)
plot_df['general_health'] = plot_df.general_health.astype(genhealth_order)

# plot
alt.Chart(plot_df).mark_bar().encode(
    x = alt.X('general_health', 
              sort = ['general_health'],
              title = 'Self-rated general health'),
    y = alt.Y('adverse_conditions',
              title = 'Prop. of respondents reporting ACEs'),
    color = 'sex',
    column = 'sex'
).properties(
    width = 600, 
    height = 300
)

alt.Chart(...)

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

### Q14 Visualization

Use the example above to plot the proportion of respondents reporting ACEs against smoking status for men and women.

*Hint*: you only need to modify the example by substituting smoking status for general health.

---

**Solution:**

```python
# Group by smoking status and sex
ace_smoking_sex = brfss_clean.groupby(
    ['smoking', 'sex']
)['adverse_conditions'].mean()

# Coerce indices to columns for plotting
plot_df2 = ace_smoking_sex.reset_index()

# Specify order of smoking categories
smoker_order = pd.CategoricalDtype(list(smoker_codes.values()), ordered=True)
plot_df2['smoking'] = plot_df2['smoking'].astype(smoker_order)

# Plot
alt.Chart(plot_df2).mark_bar().encode(
    x=alt.X('smoking',
            sort=list(smoker_codes.values()),
            title='Smoking Status'),
    y=alt.Y('adverse_conditions',
            title='Prop. of respondents reporting ACEs'),
    color='sex',
    column='sex'
).properties(
    width=500,
    height=300
)
```

---

## 3. Results

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q15 Associations

Write a few sentences discussing whether you have observed evidence of an association between reporting ACEs and general health, smoking status, and depression among survey respondents who answered the ACE questions.

---

**Answer**

Among the respondents who answered the ACE questions, the data reveal evidence of a positive association between reporting adverse childhood experiences and each of the three outcomes examined. Respondents with poorer self-rated general health (Fair or Poor) reported ACEs at substantially higher rates than those with Excellent or Very Good health, suggesting that childhood adversity may be linked to worse adult health outcomes. Similarly, current smokers — especially heavy smokers — showed higher ACE rates than former or never-smokers, consistent with the hypothesis that adverse childhood experiences increase the likelihood of tobacco use as a coping mechanism. The association with depression was particularly pronounced: respondents diagnosed with a depressive disorder reported ACEs at a markedly higher rate than those without such a diagnosis, indicating a strong apparent relationship between childhood adversity and adult mental health.

---

#### Q16 Generalization

Recall from the overview documentation all the care that the BRFSS dedicates to collecting a representative sample of the U.S. adult population with phone numbers. Do you think that your findings provide evidence of an association among the general public (not just the individuals survey)? Why or why not? Answer in two sentences.

---

**Answer**

Given the BRFSS's careful probability-based sampling design — using random-digit dialing to reach a representative sample of U.S. adults with telephone access — the observed associations likely reflect real patterns in the broader U.S. adult population with phone access. However, the findings cannot be generalized to all U.S. adults, since individuals without telephone access (e.g., those experiencing homelessness or in institutional settings) are systematically excluded from the sampling frame, and these groups may have higher rates of both ACEs and adverse health outcomes.

---

<div class="alert alert-block alert-info">
<b>‼️ Problem</b> 
</div>

#### Q17 Bias

What is a potential source of bias in the survey results, and how might this affect the proportions you've calculated?

---

**Answer**

One potential source of bias is **recall and social desirability bias** in the self-reported adverse childhood experience questions. Respondents may underreport sensitive childhood experiences — such as parental drug use or incarceration — due to discomfort, shame, or faded memory, leading to underestimates of ACE prevalence across all groups. If this underreporting is more pronounced among certain demographic groups (e.g., older respondents who may be less willing to discuss childhood trauma), it could artificially attenuate the estimated associations between ACEs and outcomes such as smoking and depression, making any true relationships appear weaker than they actually are.

---

## Submission

1. Save file to confirm all changes are on disk
2. Run *Kernel > Restart & Run All* to execute all code from top to bottom
3. Save file again to write any new output to disk
4. Export your notebook as a pdf (either through latex or as html before saving as a pdf).
5. Submit your pdf to Canvas.